# Associate visits from butler registry with tracts and patches

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS
- member : DESC, rubin-inkind
- **creation date** : 2026-03-25
- **last update** : 2026-05-31
- **last update** : 2026-06-24

**Goal** : Make list of visits with tract to be used for Fink-Alerts Broker to run DRP. Use butler main. 

In [ ]:
import lsst.pipe.base

print(lsst.pipe.base.__version__)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from lsst.geom import SpherePoint, degrees
import lsst.geom as geom
from lsst.geom import Point2D

from lsst.skymap import PatchInfo, Index2D

from astropy.time import Time

import gc

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
# try:
#    import ipympl  # noqa: F401
#    %matplotlib widget
#    print("ipympl found → interactive backend (%matplotlib widget)")
# except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
#    print("Install with:  pip install ipympl")


In [ ]:
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.labelsize"] = "x-large"
plt.rcParams["axes.titlesize"] = "x-large"
plt.rcParams["xtick.labelsize"] = "x-large"
plt.rcParams["ytick.labelsize"] = "x-large"

In [ ]:
mpl.rcParams.update(
    {
        "figure.figsize": (8, 5),  # default figure size
        "font.size": 14,  # taille de base
        "axes.titlesize": 18,  # titre de l'axe
        "axes.labelsize": 16,  # labels x et y
        "xtick.labelsize": 14,  # ticks x
        "ytick.labelsize": 14,  # ticks y
        "legend.fontsize": 14,  # legend text
        "legend.title_fontsize": 15,  # legend title
        "figure.titlesize": 20,  # titre global de la figure
    }
)

In [ ]:
import traceback

In [ ]:
# Define butler
from lsst.daf.butler import Butler

In [ ]:
!eups list lsst_distrib

In [ ]:
def getLostOfVisits(registry, where_clause_date):
    """
    Get exposure/visits from butler registry
    parameters:
    ==========
        registry : butler registry
        where_clause_date : specify which instrument and exposure dates in the butler registry
    """
    df_exposure = pd.DataFrame(
        {
            "id": pd.Series(dtype="int"),
            "obs_id": pd.Series(dtype="int"),
            "day_obs": pd.Series(dtype="int"),
            "seq_num": pd.Series(dtype="int"),
            "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c'est un datetime
            "time_end": pd.Series(dtype="str"),  # idem
            "type": pd.Series(dtype="str"),
            "target": pd.Series(dtype="str"),
            "filter": pd.Series(dtype="str"),
            "zenith_angle": pd.Series(dtype="float"),
            "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
            "ra": pd.Series(dtype="float"),
            "dec": pd.Series(dtype="float"),
            "skyangle": pd.Series(dtype="float"),
            "azimuth": pd.Series(dtype="float"),
            "zenith": pd.Series(dtype="float"),
            "science_program": pd.Series(dtype="str"),
            "jd": pd.Series(dtype="float"),
            "mjd": pd.Series(dtype="float"),
        }
    )

    # save the data array in rows before saving in pandas dataframe
    rows = []
    for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
        try:
            jd_start = info.timespan.begin.value
            jd_end = info.timespan.end.value
            the_Time_start = Time(jd_start, format="jd", scale="utc")
            the_Time_end = Time(jd_end, format="jd", scale="utc")
            mjd_start = the_Time_start.mjd
            mjd_end = the_Time_end.mjd
            isot_start = the_Time_start.isot
            isot_end = the_Time_end.isot

            if count == 0:
                print("===== Time Conversion Debug Info =====")
                print(f"JD start      : {jd_start} (type: {type(jd_start)})")
                print(f"JD end        : {jd_end} (type: {type(jd_end)})")
                print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
                print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
                print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
                print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
                print("=======================================")

            # put row in a dictionnary before stacking
            row = {
                "id": info.id,
                "obs_id": info.obs_id,
                "day_obs": info.day_obs,
                "seq_num": info.seq_num,
                "time_start": isot_start,
                "time_end": isot_end,
                "type": info.observation_type,
                "target": info.target_name,
                "filter": info.physical_filter,
                "zenith_angle": info.zenith_angle,
                "expos": info.exposure_time,  # Exemple : adapter selon ton objet
                "ra": info.tracking_ra,
                "dec": info.tracking_dec,
                "skyangle": info.sky_angle,
                "azimuth": info.azimuth,
                "zenith": info.zenith_angle,
                "science_program": info.science_program,
                "jd": float(jd_start),
                "mjd": float(mjd_start),
            }
            rows.append(row)

        except ValueError as e:
            print(f"Erreur de valeur : {e}")
        except FileNotFoundError as e:
            print(f"Fichier introuvable : {e}")
        except Exception as e:
            print(f"Erreur inattendue : {type(e).__name__} - {e}")
            print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
            traceback.print_exc()  # displays the full stack trace

    # Final DataFrame creation
    df_exposure = pd.DataFrame(rows)
    df_science = df_exposure[df_exposure.type == "science"]
    df_science.reset_index(drop=True, inplace=True)

    return df_science

## Configuration

In [ ]:
CONE_RADIUS_DEG = 1.8
MIN_NSOURCES = 500

REPO = "main"
COLLECTION = "LSSTCam/defaults"
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
DATE_STOP = 20260630
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}" f" and day_obs <= {DATE_STOP}"

TRACT_SEARCH_RADIUS_DEG = 1.8
DDF_RADIUS = TRACT_SEARCH_RADIUS_DEG

DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}


print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")


# Visits filename
# FILENAME_VISITS = "visitTable-2025041500138-2026010800515_N34276_NoTracts.csv"

# FLAGS
FLAG_FETCH_VISITSEXPOSURES = True
FLAG_SAVE_VISITSFILE = True

# Output directory for DRP data
OUTPUT_DIR = "data_DP2_VisitsAndTracts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Access to Butler Butler registry

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

## Create a skymap object

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception as inst:
    print(type(inst))  # the exception type
    print(inst.args)  # arguments stored in .args
    print(inst)  # __str__ allows args to be printed directly,
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)

print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")
# skymap = butler.get("skyMap", skymap=skymapName, collections=collection_validation)

In [ ]:
print(WHERE_CLAUSE_DATE)

## Retrieve the science visits in order to match the visitid with the MJD in forced source photometry
- `visitid = id`
- `MJD = mjd `

In [ ]:
if FLAG_FETCH_VISITSEXPOSURES:
    visitTable = getLostOfVisits(registry, where_clause_date=WHERE_CLAUSE_DATE)
    print(visitTable.head(1))
    visitMin, visitMax, Nvisits = visitTable["id"].min(), visitTable["id"].max(), len(visitTable)
    print(f"visitMin = {visitMin},visitMax = {visitMax}, Nvisits = {Nvisits}")
    visit_filename = f"visitTable-{visitMin}-{visitMax}_N{Nvisits}_NoTracts.csv"
    print(f"filename_visits = {visit_filename}")

    if FLAG_SAVE_VISITSFILE:
        visit_fullfilename = os.path.join(OUTPUT_DIR, visit_filename)
        visitTable.to_csv(visit_fullfilename)
    # del visitTable
    # gc.collect()

In [ ]:
visitTable["band"] = visitTable["filter"].apply(lambda x: x.split("_")[0])

In [ ]:
visitTable = visitTable.sort_values(by="id")
visitTable.head()

## Add Tract-Patches

In [ ]:
def get_tract_patch(row, skymap, raname="ra", decname="dec"):
    if pd.isna(row[raname]) or pd.isna(row[decname]):
        return pd.Series(
            {
                "tract": None,
                "patch": None,
                "patch_str": None,
                "tract_bbox": None,
                "tract_ra_corners": None,
                "tract_dec_corners": None,
            }
        )

    target_point = SpherePoint(row[raname], row[decname], degrees)

    tract_info = skymap.findTract(target_point)
    patch_info = tract_info.findPatch(target_point)

    # IDs
    tractNbSel = tract_info.getId()
    patchNbSel = patch_info.getSequentialIndex()
    patch_index_str = f"{patch_info.getIndex()[0]},{patch_info.getIndex()[1]}"

    # =========================
    # BBOX en pixels
    # =========================
    bbox = tract_info.getBBox()
    bbox_tuple = (bbox.getMinX(), bbox.getMinY(), bbox.getMaxX(), bbox.getMaxY())

    # =========================
    # Corners in sky coordinates (RA/Dec)
    # =========================
    wcs = tract_info.getWcs()
    corners = bbox.getCorners()

    ra_list = []
    dec_list = []

    for corner in corners:
        corner2d = Point2D(corner)
        sky = wcs.pixelToSky(corner2d)
        ra_list.append(sky.getRa().asDegrees())
        dec_list.append(sky.getDec().asDegrees())

    return pd.Series(
        {
            "tract": tractNbSel,
            "patch": patchNbSel,
            "patch_str": patch_index_str,
            "tract_bbox": bbox_tuple,
            "tract_ra_corners": ra_list,
            "tract_dec_corners": dec_list,
        }
    )

In [ ]:
df = visitTable.copy()

In [ ]:
# compute tract-patch
df_tractpatch = df.apply(get_tract_patch, axis=1, args=(skymap,))

In [ ]:
df = pd.concat([df, df_tractpatch], axis=1)
df["tract"] = df["tract"].astype("Int64")
df["patch"] = df["patch"].astype("Int64")

In [ ]:
df.head()

In [ ]:
df.tail()

## Save in a parquet file

In [ ]:
id_min = df.id.min()
id_max = df.id.max()
nvisits = len(df)

In [ ]:
print(f"\t >> Science : {nvisits} visits ==> first visit = {id_min}, last visit = {id_max}")

In [ ]:
#! pip install --user openpyxl

In [ ]:
if FLAG_SAVE_VISITSFILE:
    output_filename_parquet = f"visitTable-{id_min}-{id_max}_N{nvisits}_WithTracts.parquet"
    output_fullfilename_parquet = os.path.join(OUTPUT_DIR, output_filename_parquet)
    df.to_parquet(output_fullfilename_parquet)

In [ ]:
print(output_fullfilename_parquet)